In [ ]:
import os

def load_env(env_path=".env"):
    if not os.path.exists(env_path):
        print(f"Warning: .env file not found at {env_path}")
        return

    print(f"Loading environment variables from {env_path}")
    with open(env_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            # Find the first '=' sign
            equal_sign_index = line.find('=')
            if equal_sign_index == -1:
                print(f"Warning: Skipping line without '=': {line}")
                continue

            key = line[:equal_sign_index].strip()
            value = line[equal_sign_index + 1:].strip()

            if key:
                os.environ[key] = value
                print(f"Set environment variable: {key}")

load_env()


In [ ]:
import os
import glob
import shlex

# Environment variables
input_filename = os.environ.get('FILENAME', "test_vocals_dry.wav")
model_filename = os.environ.get('MODEL_FILENAME', "model.sf_pkg")
model_dir = os.environ.get('MODEL_DIR', "~/DDSP-SVC/model") # "~/DDSP-SVC/exp/reflow-test"
input_dir = os.environ.get('INPUT_DIR', "~/MSST-WebUI/results")
step = os.environ.get('STEP', "auto")
ts = float(os.environ.get('TS', 0.0))
key = os.environ.get('KEY', "0")

# Setup paths
input_path = os.path.expanduser(os.path.join(input_dir, input_filename))
model_path = os.path.expanduser(os.path.join(model_dir, model_filename))
model_name_no_ext = os.path.splitext(model_filename)[0]

# Determine if we are processing a single file or a directory
if os.path.isdir(input_path):
    # Get all files (adjust extensions like .wav, .flac as needed)
    files_to_process = [f for f in glob.glob(os.path.join(input_path, "*")) if os.path.isfile(f)]
else:
    files_to_process = [input_path]

for file in files_to_process:
    # Generate output name: model_name + input_file_name
    base_file_name = os.path.basename(file)
    model_name_no_ext = model_name_no_ext + key if key != "0" else model_name_no_ext 
    out_name = f"{model_name_no_ext}_{base_file_name}"
    
    quoted_input = shlex.quote(file)
    quoted_output = shlex.quote(out_name)
    quoted_model = shlex.quote(model_path)
    print(f"Processing: {base_file_name} -> {out_name}")
    
    # Run the command
    !python main_reflow.py -i {quoted_input} -m {quoted_model} -step {step} -o {quoted_output} -ts {ts} -k {key}
